## Easy Query Chatbot for AWS Resources

Easy Query Chatbot use Streamlit as front end, AWS Bedrock for GenAI model, and Steampipe for AWS Resources query and table output.

#### Pre-requisites
This notebook requires permissions to:
- create and delete Amazon IAM roles
- create, update and delete Amazon S3 buckets
- access Amazon Bedrock
- access to Amazon OpenSearch Serverless

If running on SageMaker Studio, you should add the following managed policies to your role:
- IAMFullAccess
- AWSLambda_FullAccess
- AmazonS3FullAccess
- AmazonBedrockFullAccess
- Custom policy for Amazon OpenSearch Serverless such as:
```
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "aoss:*",
            "Resource": "*"
        }
    ]
}
```

In [1]:
%pip install -U opensearch-py
%pip install -U boto3
%pip install -U retrying

  Attempting uninstall: opensearch-py
    Found existing installation: opensearch-py 2.7.1
    Uninstalling opensearch-py-2.7.1:
      Successfully uninstalled opensearch-py-2.7.1
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 16.2 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: botocore
    Found existing installation: botocore 1.35.71
    Uninstalling botocore-1.35.71:
      Successfully uninstalled botocore-1.35.71
  Attempting uninstall: boto3
    Found existing installation: boto3 1.35.71
    Uninstalling boto3-1.35.71:
      Successfully uninstalled boto3-1.35.71
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 1.36.12 requires botocore==1.35.71, but you have botocore 1.35.75 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you 

In [2]:
# restart kernel
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import json
import os
import boto3
import pprint
from utility import create_bedrock_execution_role, create_oss_policy_attach_bedrock_execution_role, create_policies_in_oss
import random
from retrying import retry
suffix = random.randrange(200, 900)

sts_client = boto3.client('sts')
boto3_session = boto3.session.Session()
region_name = boto3_session.region_name
bedrock_agent_client = boto3_session.client('bedrock-agent', region_name=region_name)
service = 'aoss'
s3_client = boto3.client('s3')
account_id = sts_client.get_caller_identity()["Account"]
s3_suffix = f"{region_name}-{account_id}"
bucket_name = f'bedrock-kb-easyquery-{s3_suffix}' # replace it with your bucket name.
pp = pprint.PrettyPrinter(indent=2)

In [5]:
# Create S3 bucket for knowledge base data source
s3bucket = s3_client.create_bucket(
    Bucket=bucket_name,
    CreateBucketConfiguration={ 'LocationConstraint': region_name }
)
s3bucket

{'ResponseMetadata': {'RequestId': 'XYN8E52XSNKQDSRY',
  'HostId': 'noBUBeX/5ABqK9eXoLz/cErPstU0QwJPTGdjsbiyRGramdddb7QvzIjMrpdv9UCChXblRtu8gNM=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'noBUBeX/5ABqK9eXoLz/cErPstU0QwJPTGdjsbiyRGramdddb7QvzIjMrpdv9UCChXblRtu8gNM=',
   'x-amz-request-id': 'XYN8E52XSNKQDSRY',
   'date': 'Thu, 05 Dec 2024 01:57:34 GMT',
   'location': 'http://bedrock-kb-easyquery-us-west-2-126672810070.s3.amazonaws.com/',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'Location': 'http://bedrock-kb-easyquery-us-west-2-126672810070.s3.amazonaws.com/'}

## Create a vector store - OpenSearch Serverless index

### Step 1 - Create OSS policies and collection
Firt of all we have to create a vector store. In this section we will use *Amazon OpenSerach serverless.*

Amazon OpenSearch Serverless is a serverless option in Amazon OpenSearch Service. As a developer, you can use OpenSearch Serverless to run petabyte-scale workloads without configuring, managing, and scaling OpenSearch clusters. You get the same interactive millisecond response times as OpenSearch Service with the simplicity of a serverless environment. Pay only for what you use by automatically scaling resources to provide the right amount of capacity for your application—without impacting data ingestion.

In [6]:
import boto3
import time
vector_store_name = f'bedrock-easy-query-{suffix}'
index_name = f"bedrock-easy-query-index-{suffix}"
aoss_client = boto3_session.client('opensearchserverless')
bedrock_kb_execution_role = create_bedrock_execution_role(bucket_name=bucket_name)
bedrock_kb_execution_role_arn = bedrock_kb_execution_role['Role']['Arn']
bedrock_kb_execution_role_arn

'arn:aws:iam::126672810070:role/AmazonBedrockExecutionRoleForKnowledgeBase_588'

In [7]:
# create security, network and data access policies within OSS
encryption_policy, network_policy, access_policy = create_policies_in_oss(vector_store_name=vector_store_name,
                       aoss_client=aoss_client,
                       bedrock_kb_execution_role_arn=bedrock_kb_execution_role_arn)
collection = aoss_client.create_collection(name=vector_store_name,type='VECTORSEARCH')

In [8]:
pp.pprint(collection)
collection_id = collection['createCollectionDetail']['id']
host = collection_id + '.' + region_name + '.aoss.amazonaws.com'
print(host)

{ 'ResponseMetadata': { 'HTTPHeaders': { 'connection': 'keep-alive',
                                         'content-length': '314',
                                         'content-type': 'application/x-amz-json-1.0',
                                         'date': 'Thu, 05 Dec 2024 02:03:05 '
                                                 'GMT',
                                         'x-amzn-requestid': '7a39d533-56ee-45c3-8d96-5b25dacaf624'},
                        'HTTPStatusCode': 200,
                        'RequestId': '7a39d533-56ee-45c3-8d96-5b25dacaf624',
                        'RetryAttempts': 0},
  'createCollectionDetail': { 'arn': 'arn:aws:aoss:us-west-2:126672810070:collection/5bnzyoqhhf1xj9umok1l',
                              'createdDate': 1733364185123,
                              'id': '5bnzyoqhhf1xj9umok1l',
                              'kmsKeyArn': 'auto',
                              'lastModifiedDate': 1733364185123,
                             

In [9]:
# wait for collection creation
response = aoss_client.batch_get_collection(names=[vector_store_name])
# Periodically check collection status
while (response['collectionDetails'][0]['status']) == 'CREATING':
    print('Creating collection...')
    time.sleep(30)
    response = aoss_client.batch_get_collection(names=[vector_store_name])
print('\nCollection successfully created:')
print(response["collectionDetails"])


Collection successfully created:
[{'arn': 'arn:aws:aoss:us-west-2:126672810070:collection/5bnzyoqhhf1xj9umok1l', 'collectionEndpoint': 'https://5bnzyoqhhf1xj9umok1l.us-west-2.aoss.amazonaws.com', 'createdDate': 1733364185123, 'dashboardEndpoint': 'https://5bnzyoqhhf1xj9umok1l.us-west-2.aoss.amazonaws.com/_dashboards', 'id': '5bnzyoqhhf1xj9umok1l', 'kmsKeyArn': 'auto', 'lastModifiedDate': 1733364208510, 'name': 'bedrock-easy-query-561', 'standbyReplicas': 'ENABLED', 'status': 'ACTIVE', 'type': 'VECTORSEARCH'}]


In [10]:
# create oss policy and attach it to Bedrock execution role
create_oss_policy_attach_bedrock_execution_role(collection_id=collection_id,
                                                bedrock_kb_execution_role=bedrock_kb_execution_role)

Opensearch serverless arn:  arn:aws:iam::126672810070:policy/AmazonBedrockOSSPolicyForKnowledgeBase_588


## Step 2 - Create vector index

In [11]:
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
credentials = boto3.Session().get_credentials()
awsauth = auth = AWSV4SignerAuth(credentials, region_name, service)

index_name = f"bedrock-easy-query-index-{suffix}"
body_json = {
   "settings": {
      "index.knn": "true",
       "number_of_shards": 1,
       "number_of_replicas": 0,
   },
   "mappings": {
      "properties": {
         "vector": {
            "type": "knn_vector",
            "dimension": 1536,
             "method": {
                 "name": "hnsw",
                 "engine": "faiss",
                 "space_type": "innerproduct",
                 "parameters": {
                     "ef_construction": 512,
                     "m": 16
                 },
             },
         },
         "text": {
            "type": "text"
         },
         "text-metadata": {
            "type": "text"         }
      }
   }
}
# Build the OpenSearch client
oss_client = OpenSearch(
    hosts=[{'host': host, 'port': 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)
# # It can take up to a minute for data access rules to be enforced
time.sleep(60)

In [12]:
# Create index
response = oss_client.indices.create(index=index_name, body=json.dumps(body_json))
print('\nCreating index:')
print(response)
time.sleep(60) # index creation can take up to a minute


Creating index:
{'acknowledged': True, 'shards_acknowledged': True, 'index': 'bedrock-easy-query-index-561'}


#### Upload Knowledge Base data to S3 Bucket

Directory ./tables contains Steampipe documentations about how to write SQL queries for different use-cases. This is where Bedrock agent retrieve accurate information and create SQL queries based on user's questions. We upload all these documents into a S3 bucket before ingest into Bedrock knowledge base. 

In [13]:
data_root = "./tables/"

# Upload data to s3
s3_client = boto3.client("s3")
def uploadDirectory(path,bucket_name):
        for root,dirs,files in os.walk(path):
            for file in files:
                s3_client.upload_file(os.path.join(root,file),bucket_name,file)

uploadDirectory(data_root, bucket_name)

## Create Knowledge Base
Steps:
- initialize Open search serverless configuration which will include collection ARN, index name, vector field, text field and metadata field.
- initialize chunking strategy, based on which KB will split the documents into pieces of size equal to the chunk size mentioned in the `chunkingStrategyConfiguration`.
- initialize the s3 configuration, which will be used to create the data source object later.
- initialize the Titan embeddings model ARN, as this will be used to create the embeddings for each of the text chunks.

In [14]:
opensearchServerlessConfiguration = {
            "collectionArn": collection["createCollectionDetail"]['arn'],
            "vectorIndexName": index_name,
            "fieldMapping": {
                "vectorField": "vector",
                "textField": "text",
                "metadataField": "text-metadata"
            }
        }

chunkingStrategyConfiguration = {
    "chunkingStrategy": "SEMANTIC",
    "semanticChunkingConfiguration": {
        "breakpointPercentileThreshold": 95,
        "bufferSize": 0,
        "maxTokens": 8192
    }
}

s3Configuration = {
    "bucketArn": f"arn:aws:s3:::{bucket_name}",
    # "inclusionPrefixes":["*.*"] # you can use this if you want to create a KB using data within s3 prefixes.
}

embeddingModelArn = f"arn:aws:bedrock:{region_name}::foundation-model/amazon.titan-embed-text-v1"

name = f"bedrock-steampipe-table-knowledge-base-{suffix}"
description = "Use this Knowledgebase to fetch steam pipe schema data relavant to the user query"
roleArn = bedrock_kb_execution_role_arn


Provide the above configurations as input to the `create_knowledge_base` method, which will create the Knowledge base.

In [15]:
# Create a KnowledgeBase
from retrying import retry

@retry(wait_random_min=1000, wait_random_max=2000,stop_max_attempt_number=7)
def create_knowledge_base_func():
    create_kb_response = bedrock_agent_client.create_knowledge_base(
        name = name,
        description = description,
        roleArn = roleArn,
        knowledgeBaseConfiguration = {
            "type": "VECTOR",
            "vectorKnowledgeBaseConfiguration": {
                "embeddingModelArn": embeddingModelArn
            }
        },
        storageConfiguration = {
            "type": "OPENSEARCH_SERVERLESS",
            "opensearchServerlessConfiguration":opensearchServerlessConfiguration
        }
    )
    return create_kb_response["knowledgeBase"]

In [16]:
try:
    kb = create_knowledge_base_func()
except Exception as err:
    print(f"{err=}, {type(err)=}")

In [17]:
pp.pprint(kb)

{ 'createdAt': datetime.datetime(2024, 12, 5, 2, 10, 56, 443491, tzinfo=tzutc()),
  'description': 'Use this Knowledgebase to fetch steam pipe schema data '
                 'relavant to the user query',
  'knowledgeBaseArn': 'arn:aws:bedrock:us-west-2:126672810070:knowledge-base/8VQV1ARR8Y',
  'knowledgeBaseConfiguration': { 'type': 'VECTOR',
                                  'vectorKnowledgeBaseConfiguration': { 'embeddingModelArn': 'arn:aws:bedrock:us-west-2::foundation-model/amazon.titan-embed-text-v1'}},
  'knowledgeBaseId': '8VQV1ARR8Y',
  'name': 'bedrock-steampipe-table-knowledge-base-561',
  'roleArn': 'arn:aws:iam::126672810070:role/AmazonBedrockExecutionRoleForKnowledgeBase_588',
  'status': 'CREATING',
  'storageConfiguration': { 'opensearchServerlessConfiguration': { 'collectionArn': 'arn:aws:aoss:us-west-2:126672810070:collection/5bnzyoqhhf1xj9umok1l',
                                                                   'fieldMapping': { 'metadataField': 'text-metadata',
  

In [18]:
# Get KnowledgeBase 
get_kb_response = bedrock_agent_client.get_knowledge_base(knowledgeBaseId = kb['knowledgeBaseId'])

Next we need to create a data source, which will be associated with the knowledge base created above. Once the data source is ready, we can then start to ingest the documents.

In [19]:
# Create a DataSource in KnowledgeBase 
create_ds_response = bedrock_agent_client.create_data_source(
    name = name,
    description = description,
    knowledgeBaseId = kb['knowledgeBaseId'],
    dataSourceConfiguration = {
        "type": "S3",
        "s3Configuration":s3Configuration
    },
    vectorIngestionConfiguration = {
        "chunkingConfiguration": chunkingStrategyConfiguration
    }
)
ds = create_ds_response["dataSource"]
pp.pprint(ds)

{ 'createdAt': datetime.datetime(2024, 12, 5, 2, 11, 23, 57302, tzinfo=tzutc()),
  'dataDeletionPolicy': 'DELETE',
  'dataSourceConfiguration': { 's3Configuration': { 'bucketArn': 'arn:aws:s3:::bedrock-kb-easyquery-us-west-2-126672810070'},
                               'type': 'S3'},
  'dataSourceId': 'SHNIX5C1FA',
  'description': 'Use this Knowledgebase to fetch steam pipe schema data '
                 'relavant to the user query',
  'knowledgeBaseId': '8VQV1ARR8Y',
  'name': 'bedrock-steampipe-table-knowledge-base-561',
  'status': 'AVAILABLE',
  'updatedAt': datetime.datetime(2024, 12, 5, 2, 11, 23, 57302, tzinfo=tzutc()),
  'vectorIngestionConfiguration': { 'chunkingConfiguration': { 'chunkingStrategy': 'SEMANTIC',
                                                               'semanticChunkingConfiguration': { 'breakpointPercentileThreshold': 95,
                                                                                                  'bufferSize': 0,
                 

In [20]:
# Get DataSource 
bedrock_agent_client.get_data_source(knowledgeBaseId = kb['knowledgeBaseId'], dataSourceId = ds["dataSourceId"])

{'ResponseMetadata': {'RequestId': '2c8db603-0be1-4ed8-8e6f-5ebef4de40f9',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 05 Dec 2024 02:11:30 GMT',
   'content-type': 'application/json',
   'content-length': '687',
   'connection': 'keep-alive',
   'x-amzn-requestid': '2c8db603-0be1-4ed8-8e6f-5ebef4de40f9',
   'x-amz-apigw-id': 'CS7I-HabvHcEDDQ=',
   'x-amzn-trace-id': 'Root=1-67510bd2-6e7a8714510cc56e76bef48b'},
  'RetryAttempts': 0},
 'dataSource': {'createdAt': datetime.datetime(2024, 12, 5, 2, 11, 23, 57302, tzinfo=tzutc()),
  'dataDeletionPolicy': 'DELETE',
  'dataSourceConfiguration': {'s3Configuration': {'bucketArn': 'arn:aws:s3:::bedrock-kb-easyquery-us-west-2-126672810070'},
   'type': 'S3'},
  'dataSourceId': 'SHNIX5C1FA',
  'description': 'Use this Knowledgebase to fetch steam pipe schema data relavant to the user query',
  'knowledgeBaseId': '8VQV1ARR8Y',
  'name': 'bedrock-steampipe-table-knowledge-base-561',
  'status': 'AVAILABLE',
  'updatedAt': datetime.date

### Start ingestion job
Once the KB and data source is created, we can start the ingestion job.
During the ingestion job, KB will fetch the documents in the data source, pre-process it to extract text, chunk it based on the chunking size provided, create embeddings of each chunk and then write it to the vector database, in this case OSS.

In [21]:
# Start an ingestion job
start_job_response = bedrock_agent_client.start_ingestion_job(knowledgeBaseId = kb['knowledgeBaseId'], dataSourceId = ds["dataSourceId"])
job = start_job_response["ingestionJob"]
pp.pprint(job)
# Get job 
while(job['status']!='COMPLETE' ):
  get_job_response = bedrock_agent_client.get_ingestion_job(
      knowledgeBaseId = kb['knowledgeBaseId'],
        dataSourceId = ds["dataSourceId"],
        ingestionJobId = job["ingestionJobId"]
  )
  job = get_job_response["ingestionJob"]
pp.pprint(job)
time.sleep(40)

{ 'dataSourceId': 'SHNIX5C1FA',
  'ingestionJobId': 'PKBQDMKMY3',
  'knowledgeBaseId': '8VQV1ARR8Y',
  'startedAt': datetime.datetime(2024, 12, 5, 2, 11, 36, 289499, tzinfo=tzutc()),
  'statistics': { 'numberOfDocumentsDeleted': 0,
                  'numberOfDocumentsFailed': 0,
                  'numberOfDocumentsScanned': 0,
                  'numberOfMetadataDocumentsModified': 0,
                  'numberOfMetadataDocumentsScanned': 0,
                  'numberOfModifiedDocumentsIndexed': 0,
                  'numberOfNewDocumentsIndexed': 0},
  'status': 'STARTING',
  'updatedAt': datetime.datetime(2024, 12, 5, 2, 11, 36, 289499, tzinfo=tzutc())}
{ 'dataSourceId': 'SHNIX5C1FA',
  'ingestionJobId': 'PKBQDMKMY3',
  'knowledgeBaseId': '8VQV1ARR8Y',
  'startedAt': datetime.datetime(2024, 12, 5, 2, 11, 36, 289499, tzinfo=tzutc()),
  'statistics': { 'numberOfDocumentsDeleted': 0,
                  'numberOfDocumentsFailed': 0,
                  'numberOfDocumentsScanned': 496,
         

In [22]:
kb_id = kb["knowledgeBaseId"]
pp.pprint(kb_id)

'8VQV1ARR8Y'


In [23]:
%store kb_id

Stored 'kb_id' (str)


## Test the knowledge base
### Using RetrieveAndGenerate API
Behind the scenes, RetrieveAndGenerate API converts queries into embeddings, searches the knowledge base, and then augments the foundation model prompt with the search results as context information and returns the FM-generated response to the question. For multi-turn conversations, Knowledge Bases manage short-term memory of the conversation to provide more contextual results.

The output of the RetrieveAndGenerate API includes the generated response, source attribution as well as the retrieved text chunks.

In [24]:
# try out KB using RetrieveAndGenerate API
bedrock_agent_runtime_client = boto3.client("bedrock-agent-runtime", region_name=region_name)
model_id = "anthropic.claude-3-sonnet-20240229-v1:0" # try with both claude instant as well as claude-v2. for claude v2 - "anthropic.claude-v2"
model_arn = f'arn:aws:bedrock:{region_name}::foundation-model/{model_id}'

In [70]:
time.sleep(5)
query = "Give me the query to find s3 buckets with public access"
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        'text': query
    },
    retrieveAndGenerateConfiguration={
        'type': 'KNOWLEDGE_BASE',
        'knowledgeBaseConfiguration': {
            'knowledgeBaseId': kb_id,
            'modelArn': model_arn
        }
    },
)

generated_text = response['output']['text']

print(generated_text)

To find AWS S3 buckets that allow public access, you can use the following SQL query:

```sql
select
  name,
  block_public_acls,
  block_public_policy,
  ignore_public_acls,
  restrict_public_buckets
from
  aws_s3_bucket
where
  not block_public_acls
  or not block_public_policy
  or not ignore_public_acls
  or not restrict_public_buckets;
```

This query selects the bucket name and the public access block settings from the `aws_s3_bucket` table. The `where` clause filters for buckets where any of the public access block settings (`block_public_acls`, `block_public_policy`, `ignore_public_acls`, `restrict_public_buckets`) are set to `false` or `0`, indicating that public access is allowed.


## Create Steampipe Server 

Launching an EC2 instance with UserData, which install, configure, and hosting the Steampipe tables

In [26]:
# Initialize the EC2 client
ec2_client = boto3.client('ec2', region_name=region_name)


In [27]:
# Create a new VPC
vpc_response = ec2_client.create_vpc(CidrBlock='10.0.0.0/16')
vpc_id = vpc_response['Vpc']['VpcId']

# Tag the VPC
ec2_client.create_tags(Resources=[vpc_id], Tags=[{'Key': 'Name', 'Value': 'SteampipeVPC'}])

print(f"Created VPC with ID: {vpc_id}")

Created VPC with ID: vpc-005014d858ab070ab


In [28]:
# Create a subnet in the VPC
subnet_response = ec2_client.create_subnet(
    VpcId=vpc_id,
    CidrBlock='10.0.1.0/24',
    AvailabilityZone='us-west-2a'
)
subnet_id = subnet_response['Subnet']['SubnetId']

# Tag the subnet
ec2_client.create_tags(Resources=[subnet_id], Tags=[{'Key': 'Name', 'Value': 'SteampipeSubnet'}])

print(f"Created subnet with ID: {subnet_id}")


Created subnet with ID: subnet-0056d3c8b88fdf13b


In [29]:
# Create an Internet Gateway
igw_response = ec2_client.create_internet_gateway()
igw_id = igw_response['InternetGateway']['InternetGatewayId']

# Attach the Internet Gateway to the VPC
ec2_client.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)

print(f"Created and attached Internet Gateway with ID: {igw_id}")

Created and attached Internet Gateway with ID: igw-0d534e6a428b16af1


In [30]:
# Create a route table and a route to the Internet Gateway
route_table_response = ec2_client.create_route_table(VpcId=vpc_id)
route_table_id = route_table_response['RouteTable']['RouteTableId']

ec2_client.create_route(
    RouteTableId=route_table_id,
    DestinationCidrBlock='0.0.0.0/0',
    GatewayId=igw_id
)

# Associate the route table with the subnet
ec2_client.associate_route_table(RouteTableId=route_table_id, SubnetId=subnet_id)

print(f"Created and configured route table with ID: {route_table_id}")

Created and configured route table with ID: rtb-04b5313db19653171


In [31]:
# Create a security group
sg_response = ec2_client.create_security_group(
    GroupName='SteampipeSG',
    Description='Security group for Steampipe server',
    VpcId=vpc_id
)
sg_id = sg_response['GroupId']

# Add inbound rule to allow SSH (port 22) from anywhere
ec2_client.authorize_security_group_ingress(
    GroupId=sg_id,
    IpPermissions=[
        {
            'IpProtocol': 'tcp',
            'FromPort': 22,
            'ToPort': 22,
            'IpRanges': [{'CidrIp': '0.0.0.0/0'}]
        },
        {
            'IpProtocol': '-1',  # All traffic
            'FromPort': -1,
            'ToPort': -1,
            'UserIdGroupPairs': [{'GroupId': sg_id}]  # Self reference
        }
    ]
)

print(f"Created security group with ID: {sg_id}")

# Wait for VPC to be available
ec2_client.get_waiter('vpc_available').wait(VpcIds=[vpc_id])

print("VPC is now available")

Created security group with ID: sg-065d972e773468588
VPC is now available


In [71]:
# Now you can use these IDs to launch your EC2 instance
print(f"VPC ID: {vpc_id}")
print(f"Subnet ID: {subnet_id}")
print(f"Security Group ID: {sg_id}")

VPC ID: vpc-005014d858ab070ab
Subnet ID: subnet-0056d3c8b88fdf13b
Security Group ID: sg-065d972e773468588


In [143]:
# get latest ami for AL23
def get_latest_al2023_ami():
    ssm_client = boto3.client('ssm')
    
    response = ssm_client.get_parameter(
        Name='/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-6.1-x86_64'
    )
    
    return response['Parameter']['Value']
# Example usage
ami_id = get_latest_al2023_ami()
print(f"Latest Amazon Linux 2023 AMI ID: {ami_id}")

Latest Amazon Linux 2023 AMI ID: ami-055e3d4f0bbeb5878


In [148]:
# Define user data script
user_data_script = """#!/bin/bash -xe
echo "hello world"
sudo /bin/sh -c "$(curl -fsSL https://steampipe.io/install/steampipe.sh)"
sudo -u ec2-user bash << EOF
whoami
steampipe plugin install aws
export STEAMPIPE_DATABASE_PASSWORD=my_password_123
steampipe service start --show-password
EOF
"""

# Define instance parameters 'ImageId': 'ami-06249cd25f9f88c1c', 
instance_params = {
    'ImageId': ami_id,  # AMI ID for us-west-2
    'InstanceType': 't3.large',
    'MinCount': 1,
    'MaxCount': 1,
    'UserData': user_data_script,
    'NetworkInterfaces': [{
        'SubnetId': subnet_id,  # Use the subnet ID you just created
        'DeviceIndex': 0,
        'AssociatePublicIpAddress': True,
        'Groups': [sg_id]  # Use the security group ID you just created
    }],
}

In [149]:
# Launch the EC2 instance
response = ec2_client.run_instances(**instance_params)

# Get the instance ID
instance_id = response['Instances'][0]['InstanceId']

print(f"Launched EC2 instance with ID: {instance_id}")

Launched EC2 instance with ID: i-0ef365927d7f6e5fd


#### Steampipe Server by dufault will use InstanceProfile as credential and destination to scan AWS resources.

In this case, we give instance an Admin role as InstanceProfile hence the Steampipe server will host tables contains existing resources in this account. 


In [74]:
# Create an IAM client
iam_client = boto3.client('iam')

In [75]:
# Create IAM role with trust relationship for EC2
trust_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Principal': {
                'Service': 'ec2.amazonaws.com'
            },
            'Action': 'sts:AssumeRole'
        }
    ]
}

role_response = iam_client.create_role(
    RoleName='EC2SteampipeAdminRole',
    AssumeRolePolicyDocument=json.dumps(trust_policy)
)

# Attach the AdministratorAccess policy to the role
iam_client.attach_role_policy(
    RoleName='EC2SteampipeAdminRole',
    PolicyArn='arn:aws:iam::aws:policy/AdministratorAccess'
)

# Create an instance profile
instance_profile_response = iam_client.create_instance_profile(
    InstanceProfileName='EC2SteampipeAdminProfile'
)

# Add the role to the instance profile
iam_client.add_role_to_instance_profile(
    InstanceProfileName='EC2SteampipeAdminProfile',
    RoleName='EC2SteampipeAdminRole'
)

# Wait for the instance profile to be ready
import time
time.sleep(10)  # Wait for 10 seconds to ensure the instance profile is ready

# Attach the instance profile to your EC2 instance
ec2_client.associate_iam_instance_profile(
    IamInstanceProfile={
        'Name': 'EC2SteampipeAdminProfile'
    },
    InstanceId=instance_id
)

print(f"Admin role attached to EC2 instance: {instance_id}")

Admin role attached to EC2 instance: i-0f7a4630b5dc1efeb


In [105]:
# Add both tags to the EC2 instance
ec2_client.create_tags(
    Resources=[instance_id],
    Tags=[
        {'Key': 'auto-delete', 'Value': 'no'},
        {'Key': 'Name', 'Value': 'steampipe-server'}
    ]
)

print("Tags 'auto-delete: no' and 'Name: steampipe-server' added to the instance")

Tags 'auto-delete: no' and 'Name: steampipe-server' added to the instance


In [106]:
# Retrieve the instance details
response = ec2_client.describe_instances(InstanceIds=[instance_id])

# Extract the private IP address
private_ip = response['Reservations'][0]['Instances'][0]['PrivateIpAddress']

print(f"The private IP address of the instance is: {private_ip}")

The private IP address of the instance is: 10.0.1.92


#### Create Lambda Function for the Bedrock Agent Action Group

Following cells create a Bedrock action group that is backed by a Lambda function. Later on, we need to associate this action group to the Bedrock agent

In [107]:
# Initialize AWS clients
s3_client = boto3.client('s3')
lambda_client = boto3.client('lambda')
# Define variables
layer_zip_path = './lambda/layer/actiongroup_lambdalayer.zip'
s3_key = 'layers/actiongroup_lambdalayer.zip'
layer_name = 'ActionGroupLayer'

In [108]:
import os
# Upload the layer zip file to S3
if os.path.exists(layer_zip_path):
    s3_client.upload_file(layer_zip_path, bucket_name, s3_key)
    print(f"Layer zip file uploaded to s3://{bucket_name}/{s3_key}")
else:
    raise FileNotFoundError('Layer zip file does not exist.')
# Create the Lambda layer
response = lambda_client.publish_layer_version(
    LayerName=layer_name,
    Description='Action Group Lambda Layer',
    Content={
        'S3Bucket': bucket_name,
        'S3Key': s3_key
    },
    CompatibleRuntimes=['python3.9']  # Adjust as needed
)

print(f"Lambda layer created: {response['LayerArn']}")
print(f"Layer version: {response['Version']}")

Layer zip file uploaded to s3://bedrock-kb-easyquery-us-west-2-126672810070/layers/actiongroup_lambdalayer.zip
Lambda layer created: arn:aws:lambda:us-west-2:126672810070:layer:ActionGroupLayer
Layer version: 5


In [109]:
layer_arn = response['LayerVersionArn']
layer_arn

'arn:aws:lambda:us-west-2:126672810070:layer:ActionGroupLayer:5'

In [110]:
import json

# Initialize clients
iam_client = boto3.client('iam')
lambda_client = boto3.client('lambda')
# Define variables
function_name = 'SteamPipeBedrockAgentActionGroup'  # You can change this name
handler = 'lambda_function.lambda_handler'
runtime = 'python3.9'
role_name = 'LambdaExecutionRole-BedrockAgentActionGroup'
code_file_path = './lambda/lambda_function_steampipe.zip'
vpc_subnet_ids =[subnet_id]
vpc_security_group_ids = [sg_id] 


In [113]:
# Create Lambda execution IAM role
trust_relationship_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Principal': {
                'Service': 'lambda.amazonaws.com'
            },
            'Action': 'sts:AssumeRole'
        }
    ]
}

try:
    role_response = iam_client.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_relationship_policy)
    )
    print(f"IAM role created: {role_response['Role']['Arn']}")

    # Attach S3 full access policy
    iam_client.attach_role_policy(
        RoleName=role_name,
        PolicyArn='arn:aws:iam::aws:policy/AmazonS3FullAccess'
    )
    print("S3 full access policy attached to the role")

    # Attach Lambda basic execution policy
    iam_client.attach_role_policy(
        RoleName=role_name,
        PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
    )
    print("Lambda basic execution policy attached to the role")

    # Create and attach EC2 network interface policy
    ec2_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "ec2:CreateNetworkInterface",
                    "ec2:DeleteNetworkInterface",
                    "ec2:DescribeNetworkInterfaces"
                ],
                "Resource": "*"
            }
        ]
    }
    
    ec2_policy_response = iam_client.create_policy(
        PolicyName='LambdaEC2NetworkInterfacePolicy',
        PolicyDocument=json.dumps(ec2_policy)
    )
    
    iam_client.attach_role_policy(
        RoleName=role_name,
        PolicyArn=ec2_policy_response['Policy']['Arn']
    )
    print("EC2 network interface policy created and attached to the role")

    # Wait for the role to be available
    import time
    time.sleep(10)  # Wait for 10 seconds to ensure the role is available



except boto3.exceptions.ClientError as e:
    print(f"Error: {e}")


IAM role created: arn:aws:iam::126672810070:role/LambdaExecutionRole-BedrockAgentActionGroup
S3 full access policy attached to the role
Lambda basic execution policy attached to the role
EC2 network interface policy created and attached to the role


In [111]:
s3_key = 'lambda/lambda_function_steampipe.zip'
# Upload Lambda function code to S3
s3_client.upload_file(code_file_path, bucket_name, s3_key)
print(f"Lambda function code uploaded to s3://{bucket_name}/{s3_key}")

Lambda function code uploaded to s3://bedrock-kb-easyquery-us-west-2-126672810070/lambda/lambda_function_steampipe.zip


In [114]:
# Create the Lambda function
lambda_response = lambda_client.create_function(
    FunctionName=function_name,
    Runtime=runtime,
    Role=role_response['Role']['Arn'],
    Handler=handler,
    Code={
        'S3Bucket': bucket_name,
        'S3Key': s3_key
    },
    VpcConfig={
        'SubnetIds': vpc_subnet_ids,
        'SecurityGroupIds': vpc_security_group_ids
    },
    Layers=[layer_arn],
    Timeout=180,  # Set timeout to 3 minutes (180 seconds)
    Environment={
        'Variables': {
            'HOST_IP': private_ip,  # Replace with your actual HOST_IP value
            'STEAMPIPE_PSD': 'my_password_123'
        }
    }
)
print(f"Lambda function created successfully: {lambda_response['FunctionArn']}")

Lambda function created successfully: arn:aws:lambda:us-west-2:126672810070:function:SteamPipeBedrockAgentActionGroup


In [115]:
# Add resource-based policy to allow Bedrock agent to invoke the Lambda function
try:
    lambda_client.add_permission(
        FunctionName=function_name,
        StatementId='AllowBedrockAgentInvoke',
        Action='lambda:InvokeFunction',
        Principal='bedrock.amazonaws.com',
        SourceArn=f'arn:aws:bedrock:{region_name}:{account_id}:agent/*'
    )
    print("Resource-based policy added successfully to allow Bedrock agent invocation.")
except lambda_client.exceptions.ResourceConflictException:
    print("Permission already exists. Skipping addition of resource-based policy.")
except Exception as e:
    print(f"Error adding resource-based policy: {str(e)}")

Resource-based policy added successfully to allow Bedrock agent invocation.


#### Create Bedrock Agent for the Steampipe query

In [116]:
# Initialize Bedrock client
bedrock_client = boto3.client('bedrock-agent')

# Define variables
agent_name = 'SteampipeAgent'
agent_description = 'AI agent to assist with AWS operations using Steampipe queries'
instruction = """You are an AI agent designed to assist with AWS operations. Your primary function is to interpret user questions, construct steampipe application compatible  PostgreSQL queries based on the schema, and execute these queries. 
You need to CAREFULLY follow these steps: 1. Carefully review the user's question and get the constructed SQL query from the Knowledgebase. 2. After getting the query, execute the query by using "steampipe". 3. Use your judgement to use the code-interpreter if the results can be better represented in a graphical way (eg: using a piechart to show cost related questions). 

Remember: 
1. Do NOT show the generated SQL Query in the response.
2. ONLY show the output from the action in markdown format. 
3. DO NOT show the <source> or citations in the reponse. 
4. Ensure that you create column names of the SQL query based on data extracted from the Knowledgebase. DO NOT make things up."""

foundation_model = 'anthropic.claude-3-sonnet-20240229-v1:0'  # or your preferred model
lambda_function_name = function_name # The Lambda function we created earlier
knowledge_base_id = kb_id  # Replace with your existing knowledge base ID
schema_path = './lambda/steampipe-schema.json'
# Define role name and trust relationship policy for the agent execution role
agent_role_name = 'BedrockSteampipeAgentExecutionRole'
trust_relationship_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Principal': {
                'Service': 'bedrock.amazonaws.com'
            },
            'Action': 'sts:AssumeRole'
        }
    ]
}

In [117]:
# Create the policy document
policy_document = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Action': 'lambda:InvokeFunction',
            'Resource': 'arn:aws:lambda:us-west-2:126672810070:function:SteamPipeBedrockAgentActionGroup'
        }
    ]
}

In [118]:
try:
    # Create the IAM policy
    policy_response = iam_client.create_policy(
        PolicyName='SteampipeLambdaInvokePolicy',
        PolicyDocument=json.dumps(policy_document)
    )
    policy_arn = policy_response['Policy']['Arn']
    print(f"IAM policy created. Policy ARN: {policy_arn}")
except boto3.exceptions.ClientError as e:
    print(f"An error occurred while creating the policy: {e}")

IAM policy created. Policy ARN: arn:aws:iam::126672810070:policy/SteampipeLambdaInvokePolicy


In [120]:
try:
    # Read the API schema from the file
    with open(schema_path, 'r') as file:
        api_schema = json.load(file)

    # Create the IAM role for the Bedrock agent
    role_response = iam_client.create_role(
        RoleName=agent_role_name,
        AssumeRolePolicyDocument=json.dumps(trust_relationship_policy)
    )
    agent_role_arn = role_response['Role']['Arn']
    print(f'IAM Role created for Bedrock agent: {agent_role_arn}')

    # Attach necessary policies to the role
    iam_client.attach_role_policy(
        RoleName=agent_role_name,
        PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
    )
    iam_client.attach_role_policy(
        RoleName=agent_role_name,
        PolicyArn='arn:aws:iam::aws:policy/AmazonBedrockFullAccess'
    )
    iam_client.attach_role_policy(
        RoleName=agent_role_name,
        PolicyArn=policy_arn
    )
    print('Policies attached to the role')

    # Get the Lambda function ARN
    lambda_client = boto3.client('lambda')
    lambda_response = lambda_client.get_function(FunctionName=lambda_function_name)
    lambda_arn = lambda_response['Configuration']['FunctionArn']

    # Create the agent
    create_agent_response = bedrock_client.create_agent(
        agentName=agent_name,
        description=agent_description,
        instruction=instruction,
        foundationModel=foundation_model,
        idleSessionTTLInSeconds=300,
        agentResourceRoleArn=agent_role_arn
    )
    
    agent_id = create_agent_response['agent']['agentId']
    print(f"Agent created with ID: {agent_id}")

except FileNotFoundError:
    print(f"Error: API schema file not found at {schema_path}")
except json.JSONDecodeError:
    print(f"Error: Invalid JSON in the API schema file at {schema_path}")
except boto3.exceptions.ClientError as e:
    print(f"Error: {e}")

IAM Role created for Bedrock agent: arn:aws:iam::126672810070:role/BedrockSteampipeAgentExecutionRole
Policies attached to the role
Agent created with ID: NCKH81XB7Y


In [121]:
# Define the new prompt override configuration
prompt_override_config = {
    'promptConfigurations': [
        {
            'inferenceConfiguration': {
                'maximumLength': 2048,
                'stopSequences': [
                    '\n\nHuman:',
                ],
                'temperature': 0,
                'topK': 250,
                'topP': 1
            },
            'promptType': 'KNOWLEDGE_BASE_RESPONSE_GENERATION',
            'promptCreationMode': 'OVERRIDDEN',
            'basePromptTemplate': '''You are a text-to-SQL conversion agent. Your task is to convert a user's natural language query into a SQL query by analyzing search results from a knowledgebase and using the provided database schema. Follow these steps carefully:

1. You will  be provided with search results from a knowledgebase. These results contain examples and information that will help you construct the SQL query. This information is crucial for constructing a valid SQL query. The schema will be provided in the following format:

<search_results>
$search_results$
</search_results>

2. Analyze the search results and the schema carefully. Pay attention to:
   - Similar examples in the knowledgebase results
   - Table and column names in the schema that are relevant to the user's query
   - Any specific SQL functions or clauses that might be needed

3. Based on your analysis, construct a SQL query that answers the user's question. Ensure that:
   - The query uses the correct table and column names as specified in the schema
   - The query structure is similar to relevant examples from the knowledgebase results
   - The query accurately represents the user's intent
   - The query is syntactically correct and follows SQL best practices

Remember, your goal is to create an accurate SQL query that answers the user's question based on the provided schema and knowledgebase results. Do not include any information or tables that are not present in the given schema.

You must output your answer in the following format. Pay attention and follow the formatting and spacing exactly:
<answer>
<answer_part>
<text>
first answer text
</text>
<sources>
<source>source ID</source>
</sources>
</answer_part>
<answer_part>
<text>
second answer text
</text>
<sources>
<source>source ID</source>
</sources>
</answer_part>
</answer>''',
            'promptState': 'ENABLED'
        }
    ]
}

try:
    # Update the agent with the new prompt override configuration
    response = bedrock_client.update_agent(
        agentId=agent_id,
        promptOverrideConfiguration=prompt_override_config,
        agentName=agent_name,
        description=agent_description,
        instruction=instruction,
        foundationModel=foundation_model,
        idleSessionTTLInSeconds=300,
        agentResourceRoleArn=agent_role_arn
    )
    
    print(f"Agent updated successfully. New status: {response['agent']['agentStatus']}")

except boto3.exceptions.ClientError as e:
    print(f"Error updating agent: {e}")

Agent updated successfully. New status: UPDATING


In [122]:
time.sleep(10)
# Prepare the agent
prepare_agent_response = bedrock_client.prepare_agent(
    agentId=agent_id
)
print(f"Agent preparation initiated. Status: {prepare_agent_response['agentStatus']}")

# Wait for the agent to be active
while True:
    describe_agent_response = bedrock_client.get_agent(agentId=agent_id)
    if describe_agent_response['agent']['agentStatus'] == 'PREPARED':
        break
    time.sleep(10)

Agent preparation initiated. Status: PREPARING


In [123]:
import json

# Create an action group using the loaded API schema
create_action_group_response = bedrock_client.create_agent_action_group(
    agentId=agent_id,
    actionGroupName='execute-steampipe-sql-queries',
    description='Use this for action for executing SQL Queries against SteamPipe',
    actionGroupExecutor={
        'lambda': lambda_arn
    },
    apiSchema={
        'payload': json.dumps(api_schema),

    },
    agentVersion = 'DRAFT'
)
print(f"Action group created: {create_action_group_response['agentActionGroup']['actionGroupId']}")

Action group created: 0H0CUKT5TE


In [124]:
# Associate the knowledge base with the agent
bedrock_client.associate_agent_knowledge_base(
    agentId=agent_id,
    knowledgeBaseId=knowledge_base_id,
    agentVersion = 'DRAFT',
    description = 'Use this Knowledgebase to fetch steam pipe schema data relavant to the user query'
)

print(f"Knowledge base {knowledge_base_id} associated with the agent")

print("Bedrock agent creation and configuration completed successfully.")

Knowledge base 8VQV1ARR8Y associated with the agent
Bedrock agent creation and configuration completed successfully.


In [125]:
import boto3
import os

# Set up the Bedrock Agent Runtime client
bedrock_client = boto3.client('bedrock-agent')

# Define the parameters for creating the agent alias
# agent_id = 'SOXFBBHHCY'  # Replace with your actual agent ID
alias_name = 'steampipe-alias'  # Replace with your desired alias name

In [126]:
time.sleep(60)
# Prepare the agent. This is a known issue when creating 
# agent aliases in Amazon Bedrock. When creating a new alias, 
# the version created doesn't automatically include the action groups and knowledge bases from the
# agent This occurs because the agent preparation step is asynchronous and may take time to complete

prepare_agent_response = bedrock_client.prepare_agent(
    agentId=agent_id
)
print(f"Agent preparation initiated. Status: {prepare_agent_response['agentStatus']}")

# Wait for the agent to be active
while True:
    describe_agent_response = bedrock_client.get_agent(agentId=agent_id)
    if describe_agent_response['agent']['agentStatus'] == 'PREPARED':
        print(f"Agent Status: {describe_agent_response['agent']['agentStatus']}")
        break
    time.sleep(10)

Agent preparation initiated. Status: PREPARING
Agent Status: PREPARED


In [127]:
# Create the agent alias
try:
    response = bedrock_client.create_agent_alias(
        agentId=agent_id,
        agentAliasName=alias_name
    )
    
    # Print the response
    print("Agent Alias created successfully:")
    print(f"Alias ID: {response['agentAlias']['agentAliasId']}")
    print(f"Alias Name: {response['agentAlias']['agentAliasName']}")
    print(f"Alias Status: {response['agentAlias']['agentAliasStatus']}")

except Exception as e:
    print(f"An error occurred: {str(e)}")
time.sleep(10)
agent_alias_id = response['agentAlias']['agentAliasId']

Agent Alias created successfully:
Alias ID: JYTRBLDHRG
Alias Name: steampipe-alias
Alias Status: CREATING


## Deploying Front-end Streamlit -> Lambda Function that invokes Agent

The Lambda function used to invoke Bedrock agent for front-end chatbot is configured nad deployed via CloudFormation stack. 
CloudFormation template is in folder /cfn. 

In [128]:
# Upload the Lambda Layer to the same s3 bucket's lambdalayer folder
s3_client = boto3.client('s3')
lambda_client = boto3.client('lambda')
# Define variables
layer_zip_path = './lambda/layer/knowledgebase_lambdalayer.zip'
s3_key = 'layers/knowledgebase_lambdalayer.zip'
layer_name = 'KnowledgeBaseLambdaLayer'

In [129]:
import os
# Upload the layer zip file to S3
if os.path.exists(layer_zip_path):
    s3_client.upload_file(layer_zip_path, bucket_name, s3_key)
    print(f"Layer zip file uploaded to s3://{bucket_name}/{s3_key}")
else:
    raise FileNotFoundError('Layer zip file does not exist.')

Layer zip file uploaded to s3://bedrock-kb-easyquery-us-west-2-126672810070/layers/knowledgebase_lambdalayer.zip


In [130]:
import boto3
import json

# Initialize CloudFormation client
cfn_client = boto3.client('cloudformation')

# Path to your CloudFormation template
template_path = './cfn/DeployChatLambdaFunction.yaml'

# Read the template file
with open(template_path, 'r') as template_file:
    template_body = template_file.read()

# Stack name
stack_name = 'ChatLambdaFunctionStack'  # You can change this to your preferred stack name

# Parameters (you can modify these values as needed)
parameters = [
    {
        'ParameterKey': 'AgentID',
        'ParameterValue': agent_id
    },
    {
        'ParameterKey': 'AgentAliasID',
        'ParameterValue': agent_alias_id
    },
    {
        'ParameterKey': 'LambdaLayerS3BucketName',
        'ParameterValue': bucket_name
    }
]

# Create the stack
try:
    response = cfn_client.create_stack(
        StackName=stack_name,
        TemplateBody=template_body,
        Parameters=parameters,
        Capabilities=['CAPABILITY_IAM', 'CAPABILITY_NAMED_IAM']  # Add this if your template creates IAM resources
    )
    
    print(f"Stack creation initiated. Stack ID: {response['StackId']}")
    
    # Wait for the stack to complete
    waiter = cfn_client.get_waiter('stack_create_complete')
    print("Waiting for stack to be created...")
    waiter.wait(StackName=stack_name)
    
    print("Stack creation completed successfully!")

except Exception as e:
    print(f"Error creating stack: {str(e)}")

# # Optionally, you can describe the stack to get its outputs
# try:
#     describe_response = cfn_client.describe_stacks(StackName=stack_name)
#     outputs = describe_response['Stacks'][0]['Outputs']
#     print("\nStack Outputs:")
#     for output in outputs:
#         print(f"{output['OutputKey']}: {output['OutputValue']}")
# except Exception as e:
#     print(f"Error describing stack: {str(e)}")


Stack creation initiated. Stack ID: arn:aws:cloudformation:us-west-2:126672810070:stack/ChatLambdaFunctionStack/8ca6ee70-b35f-11ef-9aac-06529b824455
Waiting for stack to be created...
Stack creation completed successfully!


### Test the Contextual Chatbot Application

To test the chatbot application, complete the following steps:

1. Open a new terminal or a command line window on your machine.
2. Run the following command to install the AWS SDK for Python (Boto3). Boto3 makes it straightforward to integrate a Python application, library, or script with AWS services.
```
pip install boto3
```
3. Run the following command to install and set up a local Python development environment to run the Streamlit application:
```
pip install streamlit
```
4. Navigate to the /streamlit folder in the code base you cloned earlier.
5. Run the following command to instantiate the chatbot application:
```
python -m streamlit run chatbot.py
```
6. This should open a web based chat application powered by Streamlit in your default web browser. Use this Streamlit chatbot application to post natural language questions to start the conversations powered by Bedrock Agent.

When you submit a prompt, the Streamlit app triggers the Lambda function, which invokes the Knowledge Bases RetrieveAndGenerate API to search and generate responses.

## Clean-up Resources

Make sure to run this cleanup section when you're completely done with your work to avoid incurring unnecessary costs

In [ ]:
print("Starting cleanup process...")

# 1. Delete the OpenSearch index
try:
    # Delete the index using the same index name that was created
    # The index name was constructed as: f"bedrock-easy-query-index-{suffix}"
    response = oss_client.indices.delete(index=index_name)
    print(f"Successfully deleted OpenSearch index: {index_name}")
except Exception as e:
    print(f"Error deleting OpenSearch index: {e}")

# 2. Delete the knowledge base
try:
    bedrock_agent_client = boto3.client('bedrock-agent')
    response = bedrock_agent_client.delete_knowledge_base(
        knowledgeBaseId=kb_id
    )
    print(f"Successfully deleted knowledge base: {kb_id}")
except Exception as e:
    print(f"Error deleting knowledge base: {e}")

# 3. Delete the agent
try:
    response = bedrock_agent_client.delete_agent(
        agentId=agent_id
    )
    print(f"Successfully deleted agent: {agent_id}")
except Exception as e:
    print(f"Error deleting agent: {e}")

# 2. Delete the Lambda function
try:
    lambda_client = boto3.client('lambda')
    response = lambda_client.delete_function(
        FunctionName=function_name  # This should be the name of your Lambda function
    )
    print(f"Successfully deleted Lambda function: {function_name}")
except Exception as e:
    print(f"Error deleting Lambda function: {e}")

# 1. Delete the EC2 instance
try:
    ec2_client = boto3.client('ec2')
    
    # Terminate the EC2 instance
    response = ec2_client.terminate_instances(
        InstanceIds=[instance_id]  # instance_id from when you created the Steampipe server
    )
    
    # Wait for the instance to be terminated
    waiter = ec2_client.get_waiter('instance_terminated')
    waiter.wait(
        InstanceIds=[instance_id],
        WaiterConfig={'Delay': 15, 'MaxAttempts': 40}
    )
    
    print(f"Successfully terminated EC2 instance: {instance_id}")
    
    # Delete the security group (only after instance is terminated)
    try:
        ec2_client.delete_security_group(
            GroupId=security_group_id
        )
        print(f"Successfully deleted security group: {security_group_id}")
    except Exception as e:
        print(f"Error deleting security group: {e}")
        
except Exception as e:
    print(f"Error terminating EC2 instance: {e}")

In [ ]:
# VPC Resources Cleanup
try:
    ec2_client = boto3.client('ec2')
    
    # 1. Delete VPC Endpoints (if any)
    try:
        vpc_endpoints = ec2_client.describe_vpc_endpoints(
            Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]
        )['VpcEndpoints']
        
        for endpoint in vpc_endpoints:
            ec2_client.delete_vpc_endpoints(
                VpcEndpointIds=[endpoint['VpcEndpointId']]
            )
            print(f"Deleted VPC Endpoint: {endpoint['VpcEndpointId']}")
    except Exception as e:
        print(f"Error deleting VPC endpoints: {e}")

    # 2. Delete Subnets
    try:
        subnets = ec2_client.describe_subnets(
            Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]
        )['Subnets']
        
        for subnet in subnets:
            ec2_client.delete_subnet(SubnetId=subnet['SubnetId'])
            print(f"Deleted subnet: {subnet['SubnetId']}")
    except Exception as e:
        print(f"Error deleting subnets: {e}")

    # 3. Delete Internet Gateway
    try:
        igws = ec2_client.describe_internet_gateways(
            Filters=[{'Name': 'attachment.vpc-id', 'Values': [vpc_id]}]
        )['InternetGateways']
        
        for igw in igws:
            # First detach
            ec2_client.detach_internet_gateway(
                InternetGatewayId=igw['InternetGatewayId'],
                VpcId=vpc_id
            )
            # Then delete
            ec2_client.delete_internet_gateway(
                InternetGatewayId=igw['InternetGatewayId']
            )
            print(f"Deleted Internet Gateway: {igw['InternetGatewayId']}")
    except Exception as e:
        print(f"Error deleting internet gateway: {e}")

    # 4. Delete Route Tables (except the main route table)
    try:
        route_tables = ec2_client.describe_route_tables(
            Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]
        )['RouteTables']
        
        for rt in route_tables:
            # Skip the main route table
            if not any(association.get('Main', False) for association in rt.get('Associations', [])):
                ec2_client.delete_route_table(RouteTableId=rt['RouteTableId'])
                print(f"Deleted route table: {rt['RouteTableId']}")
    except Exception as e:
        print(f"Error deleting route tables: {e}")

    # 5. Delete Security Groups (except the default one)
    try:
        security_groups = ec2_client.describe_security_groups(
            Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]
        )['SecurityGroups']
        
        for sg in security_groups:
            # Skip the default security group
            if sg['GroupName'] != 'default':
                ec2_client.delete_security_group(GroupId=sg['GroupId'])
                print(f"Deleted security group: {sg['GroupId']}")
    except Exception as e:
        print(f"Error deleting security groups: {e}")

    # 6. Finally, delete the VPC
    try:
        ec2_client.delete_vpc(VpcId=vpc_id)
        print(f"Successfully deleted VPC: {vpc_id}")
    except Exception as e:
        print(f"Error deleting VPC: {e}")

except Exception as e:
    print(f"Error in VPC cleanup process: {e}")

In [ ]:
# S3 Bucket Cleanup
try:
    s3_client = boto3.client('s3')
    s3_resource = boto3.resource('s3')
    
    # 1. Empty the bucket first (delete all objects)
    try:
        bucket = s3_resource.Bucket(bucket_name)
        
        # Delete all object versions (required for versioned buckets)
        bucket.object_versions.all().delete()
        print("Deleted all object versions from the bucket")
        
        # Delete all objects
        bucket.objects.all().delete()
        print("Deleted all objects from the bucket")
        
    except Exception as e:
        print(f"Error emptying bucket: {e}")

    # 2. Delete the empty bucket
    try:
        s3_client.delete_bucket(Bucket=bucket_name)
        print(f"Successfully deleted S3 bucket: {bucket_name}")
    except Exception as e:
        print(f"Error deleting bucket: {e}")

except Exception as e:
    print(f"Error in S3 cleanup process: {e}")